In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

sns.set_theme(style="whitegrid")

In [ ]:
df = pd.read_csv("../data/patient_clinical_data_raw.csv")

df.head()

In [ ]:
df.tail(10)


In [ ]:
rows, columns = df.shape

print(f"Number of rows    : {rows}")
print(f"Number of columns : {columns}")

In [ ]:
df.columns.tolist()

In [ ]:
df.dtypes


In [ ]:
df.info()


In [ ]:
df.describe(include="all")


In [ ]:
unique_counts = df.nunique()

unique_counts


In [ ]:
categorical_columns = df.select_dtypes(include="object").columns

for column in categorical_columns:
    print(f"\n{'=' * 60}")
    print(f"{column}")
    print(f"{'=' * 60}")
    print(df[column].value_counts(dropna=False))
    

## Part B : Data Cleaning

The dataset is checked for missing values, duplicate records, invalid ages,
and inconsistent categorical values. Appropriate cleaning techniques are
then applied while preserving the original clinical information wherever
possible.

In [ ]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": df.isnull().mean() * 100
})

missing_summary = (
    missing_summary[missing_summary["Missing_Count"] > 0]
    .sort_values("Missing_Percentage", ascending=False)
)

missing_summary

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns

numeric_columns

In [ ]:
missing_before = df[numeric_columns].isnull().sum()

missing_before

In [ ]:
for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

In [ ]:
df[numeric_columns].isnull().sum()

In [ ]:
duplicate_count = df.duplicated().sum()

print(f"Number of duplicate records: {duplicate_count}")

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

print(f"Rows after removing duplicates: {len(df)}")

In [ ]:
df["Age"].describe()

In [ ]:
invalid_age = df[
    (df["Age"] < 0) | (df["Age"] > 120)
]

invalid_age[["Patient_ID", "Age"]]


In [ ]:
invalid_age = df[
    (df["Age"] < 0) | (df["Age"] > 120)
]

invalid_age[["Patient_ID", "Age"]]

In [ ]:
valid_age_mask = df["Age"].between(0, 120)
age_median = df.loc[valid_age_mask, "Age"].median()

# Replace ages outside the reasonable 0–120 range with the median of valid ages.
df.loc[~valid_age_mask, "Age"] = np.nan
df["Age"] = df["Age"].fillna(age_median)

print(f"Median age used for invalid/missing values: {age_median:.2f}")
print(f"Invalid ages after correction: {((df['Age'] < 0) | (df['Age'] > 120)).sum()}")


In [ ]:
df["Age"].describe()

In [ ]:
df["Gender"].value_counts(dropna=False)

In [ ]:
df["Gender"] = (
    df["Gender"]
    .str.strip()
    .str.lower()
    .replace({
        "m": "Male",
        "male": "Male",
        "f": "Female",
        "female": "Female"
    })
)

In [ ]:
df["Gender"].value_counts()

In [ ]:
df["Department"].value_counts()

In [ ]:
df["Department"] = (
    df["Department"]
    .str.strip()
    .str.title()
)

In [ ]:
df["Department"].value_counts()

## Part C :  Data Transformation

New analytical features are created from the cleaned dataset to support
patient segmentation, clinical risk identification, and treatment-cost
analysis.

In [ ]:
df["Age_Group"] = pd.cut(
    df["Age"],
    bins=[-np.inf, 17, 40, 60, np.inf],
    labels=[
        "Child",
        "Young Adult",
        "Middle Age",
        "Senior"
    ]
)

In [ ]:
df["Age_Group"].value_counts()

In [ ]:
df[["Systolic", "Diastolic"]] = (
    df["Blood_Pressure"]
    .str.split("/", expand=True)
)

In [ ]:
df["Systolic"] = pd.to_numeric(
    df["Systolic"],
    errors="coerce"
)

df["Diastolic"] = pd.to_numeric(
    df["Diastolic"],
    errors="coerce"
)

In [ ]:
df[
    [
        "Blood_Pressure",
        "Systolic",
        "Diastolic"
    ]
].head(10)

In [ ]:
df["High_Risk"] = (
    (df["Glucose"] > 140)
    | (df["Cholesterol"] > 240)
    | (df["Heart_Rate"] > 100)
    | (df["Systolic"] >= 140)
    | (df["Diastolic"] >= 90)
)

In [ ]:
df["High_Risk"].value_counts()

In [ ]:
df["Cost_Per_Day"] = (
    df["Treatment_Cost"] / df["Length_of_Stay"]
)

In [ ]:
df[
    [
        "Patient_ID",
        "Treatment_Cost",
        "Length_of_Stay",
        "Cost_Per_Day"
    ]
].head(10)

## Part D Analysis

In [ ]:
average_age = df["Age"].mean()

print(f"Average patient age: {average_age:.2f} years")

In [ ]:
patients_by_department = (
    df["Department"]
    .value_counts()
)

patients_by_department

In [ ]:
plt.figure(figsize=(10, 5))

sns.countplot(
    data=df,
    x="Department",
    order=df["Department"].value_counts().index
)

plt.title("Number of Patients by Department")
plt.xlabel("Department")
plt.ylabel("Number of Patients")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../output/patient_count_by_department.png", dpi=150)
plt.show()

In [ ]:
average_cost_by_department = (
    df.groupby("Department")["Treatment_Cost"]
    .mean()
    .sort_values(ascending=False)
)

average_cost_by_department

In [ ]:
average_stay_by_diagnosis = (
    df.groupby("Diagnosis")["Length_of_Stay"]
    .mean()
    .sort_values(ascending=False)
)

average_stay_by_diagnosis

In [ ]:
top_10_cost = (
    df.nlargest(10, "Treatment_Cost")
    [
        [
            "Patient_ID",
            "Age",
            "Gender",
            "Department",
            "Diagnosis",
            "Length_of_Stay",
            "Treatment_Cost"
        ]
    ]
)

top_10_cost

In [ ]:
highest_cost_department = (
    df.groupby("Department")["Treatment_Cost"]
    .mean()
    .idxmax()
)

highest_average_cost = (
    df.groupby("Department")["Treatment_Cost"]
    .mean()
    .max()
)

print(f"Department: {highest_cost_department}")
print(f"Average Treatment Cost: {highest_average_cost:,.2f}")

In [ ]:
df["Readmission"].value_counts()

In [ ]:
readmission_rate = (
    df["Readmission"].eq("Yes").mean() * 100
)

print(f"Readmission Rate: {readmission_rate:.2f}%")

In [ ]:
readmission_by_department = (
    df.groupby("Department")["Readmission"]
    .apply(lambda x: x.eq("Yes").mean() * 100)
    .sort_values(ascending=False)
)

readmission_by_department

In [ ]:
plt.figure(figsize=(10, 5))

readmission_by_department.plot(kind="bar")

plt.title("Readmission Rate by Department")
plt.xlabel("Department")
plt.ylabel("Readmission Rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../output/readmission_rate_by_department.png", dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

readmission_by_department.plot(kind="bar")

plt.title("Readmission Rate by Department")
plt.xlabel("Department")
plt.ylabel("Readmission Rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
diagnosis_health = (
    df.groupby("Diagnosis")[
        ["Cholesterol", "Glucose"]
    ]
    .mean()
    .sort_values("Glucose", ascending=False)
)

diagnosis_health

In [ ]:
diagnosis_health = (
    df.groupby("Diagnosis")[
        ["Cholesterol", "Glucose"]
    ]
    .mean()
    .sort_values("Glucose", ascending=False)
)

diagnosis_health

In [ ]:
high_glucose = df[
    df["Glucose"] > 140
]

high_glucose[
    [
        "Patient_ID",
        "Age",
        "Department",
        "Diagnosis",
        "Glucose"
    ]
].sort_values(
    "Glucose",
    ascending=False
)

In [ ]:
print(f"Patients with high glucose: {len(high_glucose)}")

In [ ]:
high_cholesterol = df[
    df["Cholesterol"] > 240
]

high_cholesterol[
    [
        "Patient_ID",
        "Age",
        "Department",
        "Diagnosis",
        "Cholesterol"
    ]
].sort_values(
    "Cholesterol",
    ascending=False
)

In [ ]:
print(f"Patients with high cholesterol: {len(high_cholesterol)}")

In [ ]:
high_heart_rate = df[
    df["Heart_Rate"] > 100
]

high_heart_rate[
    [
        "Patient_ID",
        "Age",
        "Department",
        "Diagnosis",
        "Heart_Rate"
    ]
].sort_values(
    "Heart_Rate",
    ascending=False
)

In [ ]:
print(f"Patients with high heart rate: {len(high_heart_rate)}")

# Final Data Set check 

In [ ]:
df.info()


In [ ]:
df.isnull().sum()

In [ ]:
df.head()

In [ ]:
df.to_csv(
    "../output/patient_clinical_data_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully.")